## Паплайн для загрузки данных, предобработки, генерации прогнозов и записи в БД

In [ ]:
import pandas as pd
import numpy as np
import random
import os
import re
import requests
import subprocess

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ⏳ прогресс-бары
import tqdm as tqdm_lib
from tqdm import tqdm

# 🧠 обработка текста и NLP
import spacy
try:
    import transformers
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"], stdout=subprocess.DEVNULL)
    import transformers
    from transformers import AutoModel, AutoTokenizer    
    
# 🤖 pyTorch
import torch

# загрузка catboost
from catboost import CatBoostClassifier


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [11]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [12]:
# создаем функцию очистки текста
def _clean_single_text(text):
    return re.sub(r"[^\w\s]", " ", text.lower())

# создаем функцию предобработки текста
def preprocess_texts_optimized(texts, nlp_model_name,
                               batch_size_cpu=256,
                               num_processes_for_cleaning=-1,
                               num_processes_for_spacy_cpu=-1):
    
    print(f"🔍 Запуск предобработки для {len(texts)} текстов...")
    
    # предварительная очистка текстов
    cleaned_texts = [_clean_single_text(text) for text in tqdm(texts, desc="Очистка")]

    # spaCy и лемматизация
    nlp = None
    processed_lemmas = []
    
    # загрузка NLP модели
    print(f"🔍 Загрузка spaCy модели: '{nlp_model_name}'.")
    nlp = spacy.load(nlp_model_name)

    # используем n_process для параллелизации
    if num_processes_for_spacy_cpu == -1:
        cpu_count = os.cpu_count()
        num_processes_for_spacy_cpu = max(1, cpu_count - 1)
    
    print(f"🔍 Лемматизация будет выполнена в {num_processes_for_spacy_cpu} потоках.")
    
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=batch_size_cpu, n_process=num_processes_for_spacy_cpu), total=len(cleaned_texts), desc="Лемматизация (CPU)"):
        lemmas = [token.lemma_ for token in doc]
        processed_lemmas.append(" ".join(lemmas))
    
    print(f"🔍 Предобработка завершена. Обработано {len(processed_lemmas)} текстов.")
    return processed_lemmas

# создаем функцию для получения усредненного эмбеддинга текста
def get_embeddings_batch(texts, tokenizer, model, device, batch_size=64):
    texts = list(texts)
    embeddings = []

    print(f"🔍 Начало генерации эмбеддингов для {len(texts)} текстов на устройстве '{device}'.")
    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Переносим каждый тензор на устройство
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # берем attention mask (1 — реальные токены, 0 — паддинг)
        mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size())
        masked_embeddings = outputs.last_hidden_state * mask

        # считаем среднее только по непаддинговым токенам
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts

        embeddings.extend(mean_pooled.cpu().numpy())
    
    print(f"🔍 Генерация эмбеддингов завершена")
    
    return embeddings

# функция предобработки входного датасета в целом
def prepare_data(data, is_train, scaler=None, ipca=None):
    
    data = data.drop(
            [
                "robot_id",
                "accounts__id",
                "articles__id",
                "articles__user_id",
                "projects__id",
                "projects__user_id",
                "counterparties__id",
                "counterparties__user_id",
                "robots__user_id",
                "article_alternative_names__user_id",
            ],
            axis=1,
        )

    # поправим типы данных и заполним пропуски метками missing (для текстовых значений категорий) и 0 для пропущенных ID
    data[
        [
            "articles__parent_id",
            "projects__parent_id",
            "counterparties__parent_id",
            "robots__id",
        ]
    ] = (
        data[
            [
                "articles__parent_id",
                "projects__parent_id",
                "counterparties__parent_id",
                "robots__id",
            ]
        ]
        .fillna(0)
        .astype("int64")
    )

    data["purpose"] = data["purpose"].fillna("missing")
    data["articles__name"] = data["articles__name"].fillna("missing")
    data["projects__name"] = data["projects__name"].fillna("missing")
    data["counterparties__name"] = data["counterparties__name"].fillna("missing")

    # конвертируем дату в datetime
    data["date"] = pd.to_datetime(data["date"], yearfirst=True, errors='coerce')

    # и убираем записи из будущего и меньше нуля (и такое бывает)
    yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    data = data[data["date"] <= yesterday]
    #data = data[data["payments_amount"] > 0]

    # переименуем и поправим тип столбца с фондами
    data = data.rename(columns={"accounts__user_id": "user_id"})
    data["user_id"] = data["user_id"].fillna(0).astype("int64")

    # кодируем текстовые поля
    # сначала очищаем и лемматизируем тексты
    data["clean_purpose"] = preprocess_texts_optimized(texts=data["purpose"],nlp_model_name="ru_core_news_sm")

    # грузим модели 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
    model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
    model = model.to(device)
    model.eval()
    
    # и запускаем генерацию эмбеддингов в назначении платежа
    data["purpose_emb"] = get_embeddings_batch(data["clean_purpose"], tokenizer, model, device)

    # 1. усредняем эмбеддинг в одно число
    data["purpose_mean"] = data["purpose_emb"].apply(lambda x: float(np.mean(x)))

    # 2. выделяем три главные компоненты с предварительным масштабированием и по батчам  
    batch_size = 10_000
    
    if is_train:
        scaler = StandardScaler()
        ipca = IncrementalPCA(n_components=3)

        # обучаем скейлер
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            scaler.partial_fit(batch)

        # обучаем PCA на масштабированных данных
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler.transform(batch)
            ipca.partial_fit(batch_scaled)

    # применяем трансформацию ко всему массиву
    transformed_batches = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к эмбеддингам"):
        batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler.transform(batch)
        transformed_batches.append(ipca.transform(batch_scaled))
        
    purpose_pca_features = np.vstack(transformed_batches)

    # делаем датафрейм
    pca_column_names = [f"purpose_pca_{i+1}" for i in range(3)]
    data[pca_column_names] = purpose_pca_features

    # удалим ненужные столбцы
    data.drop(columns=["purpose", "clean_purpose", "purpose_emb"], inplace=True)

    # генерируем эмбеддинги для названий статей
    # сначала очищаем и лемматизируем тексты
    data["clean_articles__name"] = preprocess_texts_optimized(texts=data["articles__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["articles__name_emb"] = get_embeddings_batch(data["clean_articles__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["articles__name_mean"] = data["articles__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["articles__name", "clean_articles__name", "articles__name_emb"], inplace=True)

    # генерируем эмбеддинги для названий проектов
    # сначала очищаем и лемматизируем тексты
    data["clean_projects__name"] = preprocess_texts_optimized(texts=data["projects__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["projects__name_emb"] = get_embeddings_batch(data["clean_projects__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["projects__name_mean"] = data["projects__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["projects__name", "clean_projects__name", "projects__name_emb"], inplace=True)
    
    # генерируем эмбеддинги для названий доноров
    # сначала очищаем и лемматизируем тексты
    data["clean_counterparties__name"] = preprocess_texts_optimized(texts=data["counterparties__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["counterparties__name_emb"] = get_embeddings_batch(data["clean_counterparties__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["counterparties__name_mean"] = data["counterparties__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["counterparties__name", "clean_counterparties__name", "counterparties__name_emb"], inplace=True)


    # сбрасываем неактуальные столбцы
    data = data.drop(columns=[ 
        'date', 'expenditure',
        'payments_amount','user_id','account_id', 
        'contractor_id', 'article_id', 'project_id', 
        'counterpartie_id', 'donor_id', 'donor_cat_id',
        'articles__parent_id', 'projects__parent_id',
        'counterparties__parent_id', 'robots__id'])

    return data,scaler,ipca


### Загрузим и подготовим данные для прогноза

In [13]:
# загружаем новые платежи за вчера

url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

today = pd.Timestamp.today().normalize()
weekday = today.weekday()

if weekday == 6:  # в воскресенье выкачиваем все данные и дозаполняем метками uc
    start_date = ""
    end_date = ""
else:
    start_date = str((today - pd.Timedelta(days=1)).date())
    end_date = start_date


params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming",
    "date_from": start_date,
    "date_to": end_date  
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"ℹ️ Данные загружены с сервера. Количество записей: {len(data_full)}")
#data_full.to_csv("data_download.csv", index=False)

Загружено страниц: 1 стр [00:03,  3.62s/ стр]

ℹ️ Данные загружены с сервера. Количество записей: 811


In [14]:

print('Проверим пропуски по основным признакам:',
    (data_full[['purpose','articles__name','projects__name','counterparties__name']].isna().sum()))
print(
    "Количество строк, где все 3 дополнительных признака отсутствуют:",
    data_full[['articles__name','projects__name','counterparties__name']]
        .isna()
        .all(axis=1)
        .sum()
)
# сбросим строки в которых все 3 дополнительных поля отсутствуют (роботы не отработали - качество прогноза будет плохое)
data_full = data_full.dropna(subset=['articles__name', 'projects__name', 'counterparties__name'], how='all')


Проверим пропуски по основным признакам: purpose                   2
articles__name           77
projects__name          124
counterparties__name     93
dtype: int64
Количество строк, где все 3 дополнительных признака отсутствуют: 76


In [15]:
# предобрабатываем датасет
    
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'

scaler = joblib.load(scaler_emb_path)
ipca = joblib.load(ipca_emb_path)

data_full_prepared, _, _ = prepare_data(data_full, is_train=False, scaler=scaler, ipca=ipca)


🔍 Запуск предобработки для 735 текстов...


Очистка: 100%|██████████| 735/735 [00:00<00:00, 206803.08it/s]

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.


🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 735/735 [00:21<00:00, 34.54it/s]


...

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.
🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 735/735 [00:19<00:00, 37.06it/s]


🔍 Предобработка завершена. Обработано 735 текстов.
🔍 Начало генерации эмбеддингов для 735 текстов на устройстве 'cpu'.


Генерация эмбеддингов:   0%|          | 0/12 [00:00<?, ?it/s]/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Генерация эмбеддингов: 100%|██████████| 12/12 [00:01<00:00,  6.82it/s]

🔍 Генерация эмбеддингов завершена


### Генерируем прогнозы

#### catboost

In [ ]:
best_model_cb = CatBoostClassifier()
best_model_cb.load_model("model_cb.cbm")
print('Параметры загруженной модели: ', best_model_cb.get_params())

features = [
    'purpose_mean', 'purpose_pca_1', 'purpose_pca_2', 'purpose_pca_3',
    'articles__name_mean', 'projects__name_mean', 'counterparties__name_mean'
]

data_full_prepared['uc__uc_id'] = best_model_cb.predict(data_full_prepared[features])


Параметры загруженной модели:  {'random_strength': 5, 'border_count': 256, 'od_wait': 10, 'verbose': 100, 'iterations': 1000, 'auto_class_weights': 'Balanced', 'loss_function': 'MultiClass', 'l2_leaf_reg': 9, 'task_type': 'GPU', 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.05, 'random_seed': 42}


In [ ]:
display(data_full.head(),data_full.tail())

In [ ]:
display(data_full_prepared.head(),data_full_prepared.tail())

,id,uc__uc_id,purpose_mean,purpose_pca_1,purpose_pca_2,purpose_pca_3,articles__name_mean,projects__name_mean,counterparties__name_mean,uc__uc_id_cb
0,1180158,None,-0.000798,-18.369203,14.133872,3.413240,-0.000094,-0.000592,-0.000312,5
1,1180961,None,-0.001618,13.728141,-0.301073,-8.991203,0.001444,0.001069,-0.000172,1
2,1180962,None,-0.001436,13.785214,-0.536308,-8.691223,0.001444,0.001069,-0.000172,1
3,1180963,None,-0.001436,13.785214,-0.536308,-8.691223,0.001444,0.001069,-0.000172,1
4,1180964,None,-0.001962,14.947406,7.133296,-6.165394,0.001074,0.001069,-0.000172,1


In [ ]:
# генерируем запрос на запись в БД
payload = {
    "items": [
        {"payment_id": int(row.id), "uc_id": int(row.uc__uc_id)}
        for row in data_full_prepared.itertuples(index=False)
    ]
}

In [19]:
# отправляем запрос на запись в БД
url_up = "https://api.lemonpie.tech/api/payments/article-u"
headers = {
    "Authorization": "Bearer YOUR_API_TOKEN",
    "Content-Type": "application/json",
}

response = requests.post(url_up, json=payload, headers=headers)
print(response.status_code, response.text)

200 {"status":"ok","inserted":735,"updated":0,"total":735}
